In [2]:
#import Pkg; Pkg.add("MathOptInterface")
#using Pkg
#Pkg.add("TimeZones")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed MathOptInterface ─ v1.46.0
    Updating `~/.julia/environments/v1.11/Project.toml`
  [b8f27783] + MathOptInterface v1.46.0
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [b8f27783] ↑ MathOptInterface v1.42.1 ⇒ v1.46.0
Precompiling project...
  18083.0 ms  ✓ MathOptInterface
  1 dependency successfully precompiled in 19 seconds. 326 already precompiled.
  1 dependency precompiled but a different version is currently loaded. Restart julia to access the new version. Otherwise, loading dependents of this package may trigger further precompilation to work with the unexpected version.


# Optimization

In [5]:
#############################
# Solar–Battery 7-Day Optimization
# Deterministic (point forecast) version
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates
using Statistics




# Choose a solver you have installed:
using Gurobi
 # LP solver; fine since robust_price=false in this script

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64          # MWh - energy capacity
    P_charge_max::Float64   # MW  - max charging power
    P_discharge_max::Float64 # MW  - max discharging power
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64           # MWh - initial state of charge
    k_deg::Float64          # $/MWh discharged (degradation cost)
    Δt::Float64             # hours per time step (usually 1.0)
end



function build_battery_model(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_price::Bool=false,
    Γ_price::Float64=0.0
)
    T = length(prices)
    @assert length(solar) == T "prices and solar must have same length"

    # Pick solver here
    model = Model(Gurobi.Optimizer)

    # Decision variables
    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)    # MW
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max) # MW
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)           # MWh
    @variable(model, 0 <= grid_sell[1:T])                                   # MWh

    # If robust_price is turned on, add variable & SOC constraint
    if robust_price
        @variable(model, z >= 0)  # upper bound on ||grid_sell||_2
        @constraint(model, [z; grid_sell] in MOI.SecondOrderCone())
    end

    # Initial SoC
    @constraint(model, SoC[0] == params.SoC0)

    # Battery dynamics + solar coupling + export balance
    for t in 1:T
        # State-of-charge evolution
        @constraint(model,
            SoC[t] == SoC[t-1] +
                      params.eta_charge          * charge[t]   * params.Δt -
                      (1 / params.eta_discharge) * discharge[t] * params.Δt
        )

        # Charge can only come from solar (no grid charging)
        @constraint(model, charge[t] <= solar[t])

        # Export balance: solar used either to charge or export,
        # plus discharging adds to export
        @constraint(model, grid_sell[t] == solar[t] - charge[t] + discharge[t])
        # (export[t] >= 0 enforced by variable bounds)
    end

    # Objective: profit = sum_t (price_t * export_t - k_deg * discharge_t) * Δt
    if robust_price
        # Robust price revenue: sum(π̂_t e_t) - Γ * z
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                Γ_price * model[:z] -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    else
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    end

    return model
end

#############################
# Main script
#############################

function main()
    # ---------- File paths ----------
    project_dir = ".."

    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    println("Reading CAISO LMP from: ", lmp_path)
    df_lmp = CSV.read(lmp_path, DataFrame)

    println("Reading ORT solar forecasts from: ", solar_path)
    df_solar = CSV.read(solar_path, DataFrame)

    # ---------- Extract last 7 days (168 hours) ----------
    N = 168  # 7 days * 24 hours
    @assert nrow(df_solar) == N 

    # CAISO: take last 168 LMP values as "last week of year"
    @assert nrow(df_lmp) >= N "Not enough LMP rows to extract last 168 hours."
    # Shift back LMP window by 1 day (24 hours)
    shift_hours = 48  # because solar ends 2 days earlier than LMP file
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours
    
    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])
    
    @assert length(prices_all) == 168

    # ORT irradiance predictions (assumed already in MWh per hour or convertible)
    I_ref        = 1000.0          # W/m²
    P_solar_max  = 5.0             # MW, "decent sized" PV for a 5 MW / 20 MWh battery
    
    solar_irr = df_solar.prediction              # W/m²
    solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
    solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)
    
    solar_all = Vector{Float64}(solar_mw)  # MWh per hour since Δt = 1
    @assert length(solar_all) == N "Solar forecast vector must be length 168."

    # ---------- Battery parameters ----------
    params = BatteryParams(
        E_max           = 20.0,   # MWh
        P_charge_max    = 5.0,    # MW
        P_discharge_max = 5.0,    # MW
        eta_charge      = 0.95,
        eta_discharge   = 0.95,
        SoC0            = 0,   # initial SoC at start of 7-day horizon
        k_deg           = 5.0,    # $ per MWh discharged
        Δt              = 1.0     # 1-hour time steps
    )

    # ---------- Day-by-day optimization (7 days) ----------
    T_day = 24
    n_days = N ÷ T_day
    @assert n_days == 7 "Expected 7 days (168 / 24)."

    daily_profit   = zeros(n_days)
    daily_SoC_end  = zeros(n_days)

    # Store hourly decisions per day (rows = hour, cols = day)
    daily_charge     = Matrix{Float64}(undef, T_day, n_days)
    daily_discharge  = Matrix{Float64}(undef, T_day, n_days)
    daily_grid_sell     = Matrix{Float64}(undef, T_day, n_days)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*T_day + 1) : (d*T_day)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        # Use carryover SoC from previous day as SoC0
        params_day = BatteryParams(
            params.E_max,
            params.P_charge_max,
            params.P_discharge_max,
            params.eta_charge,
            params.eta_discharge,
            SoC_current,
            params.k_deg,
            params.Δt
        )

        println("\n=== Solving day $d ===")
        model = build_battery_model(prices_day, solar_day, params_day;
                                    robust_price=false)

        optimize!(model)
        term = termination_status(model)
        println("Termination status (day $d): ", term)

        if term != MOI.OPTIMAL && term != MOI.LOCALLY_SOLVED
            @warn "Model for day $d did not reach OPTIMAL; results may be invalid."
        end

        # Extract results
        daily_profit[d] = objective_value(model)

        charge    = value.(model[:charge])
        discharge = value.(model[:discharge])
        grid_sell    = value.(model[:grid_sell])
        SoC       = value.(model[:SoC])

        daily_charge[:, d]    = charge
        daily_discharge[:, d] = discharge
        daily_grid_sell[:, d]    = grid_sell

        daily_SoC_end[d] = SoC[end]
        SoC_current = SoC[end]  # carry SoC to next starting day

        println("Day $d profit: ", daily_profit[d])
        println("End-of-day SoC: ", SoC_current)
    end

    println("\n==============================")
    # Round daily and total profit
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)
    
    println("==============================")
    println("Daily profits over 7 days:")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    # If you want to save results to CSV for plotting/reporting:
    # construct a DataFrame of hourly decisions
    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)
    df_results = DataFrame(
        day   = day_id,
        hour  = hours,
        charge     = round.(vec(daily_charge), digits=2),
        discharge  = round.(vec(daily_discharge), digits=2),
        grid_sell  = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)


   

    # Load timestamps
    solar_path = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")
    df_solar = CSV.read(solar_path, DataFrame)
    
    # Fix formatting: replace space with "T"
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    
    # Now parse correctly as ZonedDateTime
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")
    
    @assert nrow(df_solar) == 168
    
    # Attach timestamps
    df_results[!, :datetime_utc] = df_solar.datetime_utc
    
    # Convert to California time
    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)
    
    # Reorder
    # Pretty string formats
    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")
    
    # Reorder and KEEP ONLY formatted versions
    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )
    
    # Save
    out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps.csv")
    CSV.write(out_path2, df_results)
    println("\nSaved WITH timestamps → ", out_path2)


    #############################
    # OPERATOR PRESCRIPTION EXPORT (JULIA)
    #############################
    
    # --- Battery constants (must match optimization) ---
    E_MAX = params.E_max
    ETA_CHARGE = params.eta_charge
    ETA_DISCHARGE = params.eta_discharge
    DT = params.Δt
    
    # ------------------------------
    # CLASSIFY ACTION PER HOUR
    # ------------------------------
    function classify_action(charge, discharge, grid_sell)
        if charge > 0.01
            return "CHARGE"
        elseif discharge > 0.01
            return "DISCHARGE"
        elseif grid_sell > 0.01
            return "SELL"
        else
            return "HOLD"
        end
    end
    
    df_results.action = [
        classify_action(df_results.charge[i],
                        df_results.discharge[i],
                        df_results.grid_sell[i])
        for i in 1:nrow(df_results)
    ]
    
    # ------------------------------
    # COMPUTE SOC TRAJECTORY
    # ------------------------------
    soc = zeros(nrow(df_results) + 1)
    soc[1] = params.SoC0
    
    for i in 1:nrow(df_results)
        soc[i+1] = soc[i] +
                   ETA_CHARGE * df_results.charge[i] * DT -
                   (1 / ETA_DISCHARGE) * df_results.discharge[i] * DT
    
        soc[i+1] = clamp(soc[i+1], 0.0, E_MAX)
    end

    df_results.soc_start = soc[1:end-1]
    df_results.soc_end   = soc[2:end]
    
    df_results.soc_pct_start = 100 .* df_results.soc_start ./ E_MAX
    df_results.soc_pct_end   = 100 .* df_results.soc_end   ./ E_MAX
    
    # ------------------------------
    # GROUP INTO CONTINUOUS ACTION BLOCKS
    # ------------------------------
    action_block = ones(Int, nrow(df_results))
    
    for i in 2:nrow(df_results)
        action_block[i] = df_results.action[i] == df_results.action[i-1] ?
                          action_block[i-1] : action_block[i-1] + 1
    end
    
    df_results.action_block = action_block
    
    summary = combine(groupby(df_results, :action_block),
        :datetime_ca => first => :start_time,
        :datetime_ca => last  => :end_time,
        :action      => first => :action,
        :charge      => mean  => :avg_charge,
        :discharge   => mean  => :avg_discharge,
        :soc_start   => first => :soc_start,
        :soc_end     => last  => :soc_end,
        :soc_pct_start => first => :soc_pct_start,
        :soc_pct_end   => last  => :soc_pct_end
    )
    
    # ------------------------------
    # ROUND FOR READABILITY
    # ------------------------------
    summary.avg_charge     = round.(summary.avg_charge, digits=2)
    summary.avg_discharge  = round.(summary.avg_discharge, digits=2)
    summary.soc_start      = round.(summary.soc_start, digits=2)
    summary.soc_end        = round.(summary.soc_end, digits=2)
    summary.soc_pct_start = round.(summary.soc_pct_start, digits=1)
    summary.soc_pct_end   = round.(summary.soc_pct_end, digits=1)
    
    # ------------------------------
    # GENERATE OPERATOR INSTRUCTIONS
    # ------------------------------
    summary.MW_instruction = [
        "From $(summary.start_time[i]) to $(summary.end_time[i]) → " *
        "$(summary.action[i]) at ~" *
        string(summary.action[i] == "CHARGE" ?
               summary.avg_charge[i] :
               summary.avg_discharge[i]) * " MW"
        for i in 1:nrow(summary)
    ]
    
    summary.SOC_instruction = [
        "From $(summary.start_time[i]) → SoC " *
        "$(summary.soc_pct_start[i])% → $(summary.soc_pct_end[i])% during " *
        "$(summary.action[i])"
        for i in 1:nrow(summary)
    ]
    
    # ------------------------------
    # EXPORT FINAL OPERATOR CSV
    # ------------------------------
    operator_path = joinpath(project_dir, "results", "battery_operator_prescription.csv")
    CSV.write(operator_path, summary)
    
    println("\n✅ Operator prescription saved to:")
    println(operator_path)
end

main()

Reading CAISO LMP from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/sdge_2024_lmp.csv
Reading ORT solar forecasts from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/ghi_predictions_new_time.csv

=== Solving day 1 ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Max
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 73 rows, 97 columns and 193 nonzeros
Model fingerprint: 0x7c36804c
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [5e+00, 6e+01]
  Bounds range     [5e+00, 2e+01]
  RHS range        [1e-03, 3e+00]
Presolve removed 50 rows and 29 columns
Presolve time: 0.00s
Presolved: 23 rows, 68 columns, 90 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.4073491e+03   7.160831e+01 

# Ellipsodial Uncertainty on Price

In [8]:
#############################
# Solar–Battery 7-Day Optimization
# Ellipsoidal Robust Pricing (2-norm)
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates

# Choose a solver you have installed:
using Gurobi

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64          # MWh - energy capacity
    P_charge_max::Float64   # MW  - max charging power
    P_discharge_max::Float64 # MW  - max discharging power
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64           # MWh - initial state of charge
    k_deg::Float64          # $/MWh discharged (degradation cost)
    Δt::Float64             # hours per time step (usually 1.0)
end

function build_battery_model(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_price::Bool=false,
    Γ_price::Float64=0.0
)
    T = length(prices)
    @assert length(solar) == T "prices and solar must have same length"

    model = Model(Gurobi.Optimizer)

    # Decision variables
    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)    # MW
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max) # MW
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)           # MWh
    @variable(model, 0 <= grid_sell[1:T])                              # MWh

    # Ellipsoidal robustification term
    if robust_price
        @variable(model, z >= 0)  # upper bound on ||grid_sell||_2
        @constraint(model, [z; grid_sell] in SecondOrderCone())

    end

    # Initial SoC
    @constraint(model, SoC[0] == params.SoC0)

    # Battery dynamics + solar coupling + export balance
    for t in 1:T
        @constraint(model,
            SoC[t] == SoC[t-1] +
                      params.eta_charge          * charge[t]   * params.Δt -
                      (1 / params.eta_discharge) * discharge[t] * params.Δt
        )

        @constraint(model, charge[t] <= solar[t])

        @constraint(model, grid_sell[t] == solar[t] - charge[t] + discharge[t])
    end

    # Objective: profit = sum_t (price_t * export_t - k_deg * discharge_t) * Δt
    if robust_price
        # Robust price revenue: sum(π̂_t e_t) - Γ * ||e||_2
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                Γ_price * model[:z] -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    else
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    end

    return model
end

#############################
# Main script — Ellipsoidal Robust
#############################

function main()
    # ---------- File paths ----------
    project_dir = ".."

    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    println("Reading CAISO LMP from: ", lmp_path)
    df_lmp = CSV.read(lmp_path, DataFrame)

    println("Reading ORT solar forecasts from: ", solar_path)
    df_solar = CSV.read(solar_path, DataFrame)

    # ---------- Extract last 7 days (168 hours) ----------
    N = 168  # 7 days * 24 hours
    @assert nrow(df_solar) == N 

    @assert nrow(df_lmp) >= N "Not enough LMP rows to extract last 168 hours."
    # Shift back LMP window by 2 days (48 hours)
    shift_hours = 48
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours
    
    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])
    @assert length(prices_all) == 168

    # ORT irradiance predictions
    I_ref        = 1000.0
    P_solar_max  = 5.0
    solar_irr = df_solar.prediction              # W/m²
    solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
    solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)
    solar_all = Vector{Float64}(solar_mw)
    @assert length(solar_all) == N

    # ---------- Battery parameters ----------
    params = BatteryParams(
        E_max           = 20.0,
        P_charge_max    = 5.0,
        P_discharge_max = 5.0,
        eta_charge      = 0.95,
        eta_discharge   = 0.95,
        SoC0            = 0.0,
        k_deg           = 5.0,
        Δt              = 1.0
    )

    # Robustness strength (tune this)
    Γ_price = 10.0   # larger = more conservative

    # ---------- Day-by-day optimization (7 days) ----------
    T_day = 24
    n_days = N ÷ T_day
    @assert n_days == 7 "Expected 7 days (168 / 24)."

    daily_profit   = zeros(n_days)
    daily_SoC_end  = zeros(n_days)

    daily_charge     = Matrix{Float64}(undef, T_day, n_days)
    daily_discharge  = Matrix{Float64}(undef, T_day, n_days)
    daily_grid_sell  = Matrix{Float64}(undef, T_day, n_days)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*T_day + 1) : (d*T_day)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        # Use carryover SoC from previous day as SoC0
        params_day = BatteryParams(
            params.E_max,
            params.P_charge_max,
            params.P_discharge_max,
            params.eta_charge,
            params.eta_discharge,
            SoC_current,
            params.k_deg,
            params.Δt
        )

        println("\n=== Solving day $d (ellipsoidal robust) ===")
        model = build_battery_model(prices_day, solar_day, params_day;
                                    robust_price=true, Γ_price=Γ_price)

        optimize!(model)
        term = termination_status(model)
        println("Termination status (day $d): ", term)

        if term != MOI.OPTIMAL && term != MOI.LOCALLY_SOLVED
            @warn "Model for day $d did not reach OPTIMAL; results may be invalid."
        end

        daily_profit[d] = objective_value(model)

        charge    = value.(model[:charge])
        discharge = value.(model[:discharge])
        grid_sell = value.(model[:grid_sell])
        SoC       = value.(model[:SoC])

        daily_charge[:, d]    = charge
        daily_discharge[:, d] = discharge
        daily_grid_sell[:, d] = grid_sell

        daily_SoC_end[d] = SoC[end]
        SoC_current = SoC[end]

        println("Day $d profit: ", daily_profit[d])
        println("End-of-day SoC: ", SoC_current)
    end

    println("\n==============================")
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)
    daily_SoC_end = round.(daily_SoC_end, digits = 2)
    
    println("Daily profits over 7 days (ellipsoid):")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    # ---------- Export results, with timestamps ----------
    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)
    df_results = DataFrame(
        day       = day_id,
        hour      = hours,
        charge    = round.(vec(daily_charge), digits=2),
        discharge = round.(vec(daily_discharge), digits=2),
        grid_sell = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results_ellipsoid.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)

    # Load timestamps
    df_solar = CSV.read(solar_path, DataFrame)
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")
    @assert nrow(df_solar) == 168

    df_results[!, :datetime_utc] = df_solar.datetime_utc

    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)

    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")

    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )
    
    out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps_ellipsoid.csv")
    CSV.write(out_path2, df_results)
    println("\nSaved WITH timestamps → ", out_path2)

    #############################
    # OPERATOR PRESCRIPTION EXPORT (JULIA)
    #############################
    
    # --- Battery constants (must match optimization) ---
    E_MAX = params.E_max
    ETA_CHARGE = params.eta_charge
    ETA_DISCHARGE = params.eta_discharge
    DT = params.Δt
    
    # ------------------------------
    # CLASSIFY ACTION PER HOUR
    # ------------------------------
    function classify_action(charge, discharge, grid_sell)
        if charge > 0.01
            return "CHARGE"
        elseif discharge > 0.01
            return "DISCHARGE"
        elseif grid_sell > 0.01
            return "SELL"
        else
            return "HOLD"
        end
    end
    
    df_results.action = [
        classify_action(df_results.charge[i],
                        df_results.discharge[i],
                        df_results.grid_sell[i])
        for i in 1:nrow(df_results)
    ]
    
    # ------------------------------
    # COMPUTE SOC TRAJECTORY
    # ------------------------------
    soc = zeros(nrow(df_results) + 1)
    soc[1] = params.SoC0
    
    for i in 1:nrow(df_results)
        soc[i+1] = soc[i] +
                   ETA_CHARGE * df_results.charge[i] * DT -
                   (1 / ETA_DISCHARGE) * df_results.discharge[i] * DT
    
        soc[i+1] = clamp(soc[i+1], 0.0, E_MAX)
    end

    df_results.soc_start = soc[1:end-1]
    df_results.soc_end   = soc[2:end]
    
    df_results.soc_pct_start = 100 .* df_results.soc_start ./ E_MAX
    df_results.soc_pct_end   = 100 .* df_results.soc_end   ./ E_MAX
    
    # ------------------------------
    # GROUP INTO CONTINUOUS ACTION BLOCKS
    # ------------------------------
    action_block = ones(Int, nrow(df_results))
    
    for i in 2:nrow(df_results)
        action_block[i] = df_results.action[i] == df_results.action[i-1] ?
                          action_block[i-1] : action_block[i-1] + 1
    end
    
    df_results.action_block = action_block
    
    summary = combine(groupby(df_results, :action_block),
        :datetime_ca => first => :start_time,
        :datetime_ca => last  => :end_time,
        :action      => first => :action,
        :charge      => mean  => :avg_charge,
        :discharge   => mean  => :avg_discharge,
        :soc_start   => first => :soc_start,
        :soc_end     => last  => :soc_end,
        :soc_pct_start => first => :soc_pct_start,
        :soc_pct_end   => last  => :soc_pct_end
    )
    
    # ------------------------------
    # ROUND FOR READABILITY
    # ------------------------------
    summary.avg_charge     = round.(summary.avg_charge, digits=2)
    summary.avg_discharge  = round.(summary.avg_discharge, digits=2)
    summary.soc_start      = round.(summary.soc_start, digits=2)
    summary.soc_end        = round.(summary.soc_end, digits=2)
    summary.soc_pct_start = round.(summary.soc_pct_start, digits=1)
    summary.soc_pct_end   = round.(summary.soc_pct_end, digits=1)
    
    # ------------------------------
    # GENERATE OPERATOR INSTRUCTIONS
    # ------------------------------
    summary.MW_instruction = [
        "From $(summary.start_time[i]) to $(summary.end_time[i]) → " *
        "$(summary.action[i]) at ~" *
        string(summary.action[i] == "CHARGE" ?
               summary.avg_charge[i] :
               summary.avg_discharge[i]) * " MW"
        for i in 1:nrow(summary)
    ]
    
    summary.SOC_instruction = [
        "From $(summary.start_time[i]) → SoC " *
        "$(summary.soc_pct_start[i])% → $(summary.soc_pct_end[i])% during " *
        "$(summary.action[i])"
        for i in 1:nrow(summary)
    ]
    
    # ------------------------------
    # EXPORT FINAL OPERATOR CSV
    # ------------------------------
    operator_path = joinpath(project_dir, "results", "ellipsoidal_battery_operator_prescription.csv")
    CSV.write(operator_path, summary)
    
    println("\n✅ Operator prescription saved to:")
    println(operator_path)
end

main()

Reading CAISO LMP from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/sdge_2024_lmp.csv
Reading ORT solar forecasts from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/ghi_predictions_new_time.csv

=== Solving day 1 (ellipsoidal robust) ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Max
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 73 rows, 98 columns and 193 nonzeros
Model fingerprint: 0xdabb85c6
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  Objective range  [5e+00, 6e+01]
  Bounds range     [5e+00, 2e+01]
  RHS range        [1e-03, 3e+00]
Presolve removed 25 rows and 1 columns
Presolve time: 0.00s
Presolved: 48 rows, 97 columns, 167 nonzeros
Presolved model ha

# Budgeted Uncertainty on Price

In [6]:
#############################
# Solar–Battery 7-Day Optimization
# Budgeted Robust Pricing (Bertsimas–Sim, 20% dev)
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates

using Gurobi

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64
    P_charge_max::Float64
    P_discharge_max::Float64
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64
    k_deg::Float64
    Δt::Float64
end

function build_battery_model_budgeted(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_price::Bool=false,
    Γ_price::Float64=0.0,
    dev_frac::Float64=0.20        # A: 20% fractional deviation
)
    T = length(prices)
    @assert length(solar) == T "prices and solar must have same length"

    model = Model(Gurobi.Optimizer)

    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max)
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)
    @variable(model, 0 <= grid_sell[1:T])

    @constraint(model, SoC[0] == params.SoC0)

    for t in 1:T
        @constraint(model,
            SoC[t] == SoC[t-1] +
                      params.eta_charge          * charge[t]   * params.Δt -
                      (1 / params.eta_discharge) * discharge[t] * params.Δt
        )
        @constraint(model, charge[t] <= solar[t])
        @constraint(model, grid_sell[t] == solar[t] - charge[t] + discharge[t])
    end

    if robust_price
        # Bertsimas–Sim downward price uncertainty:
        # p_t = p̂_t - δ_t ξ_t,  0 ≤ ξ_t ≤ 1,  Σ ξ_t ≤ Γ_price
        # Robust revenue = Σ p̂_t e_t - max Σ δ_t e_t ξ_t
        # Dual: Γ θ + Σ s_t,  θ + s_t ≥ δ_t e_t,  θ ≥ 0, s_t ≥ 0
        δ = dev_frac .* prices

        @variable(model, θ >= 0)
        @variable(model, s[1:T] >= 0)

        for t in 1:T
            @constraint(model, θ + s[t] >= δ[t] * grid_sell[t])
        end

        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                (Γ_price * θ + sum(s[t] for t in 1:T)) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    else
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    end

    return model
end

#############################
# Main script — Budgeted Robust
#############################

function main()
    project_dir = ".."

    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    println("Reading CAISO LMP from: ", lmp_path)
    df_lmp = CSV.read(lmp_path, DataFrame)

    println("Reading ORT solar forecasts from: ", solar_path)
    df_solar = CSV.read(solar_path, DataFrame)

    N = 168
    @assert nrow(df_solar) == N 

    @assert nrow(df_lmp) >= N
    shift_hours = 48
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours
    
    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])
    @assert length(prices_all) == 168

    I_ref        = 1000.0
    P_solar_max  = 5.0
    solar_irr = df_solar.prediction
    solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
    solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)
    solar_all = Vector{Float64}(solar_mw)
    @assert length(solar_all) == N

    params = BatteryParams(
        E_max           = 20.0,
        P_charge_max    = 5.0,
        P_discharge_max = 5.0,
        eta_charge      = 0.95,
        eta_discharge   = 0.95,
        SoC0            = 0.0,
        k_deg           = 5.0,
        Δt              = 1.0
    )

    # Robustness knobs
    Γ_price  = 5.0    # "budget" of bad hours
    dev_frac = 0.20   # A: 20% downward deviation

    # ---------- Day-by-day optimization (7 days) ----------
    T_day = 24
    n_days = N ÷ T_day
    @assert n_days == 7 "Expected 7 days (168 / 24)."

    daily_profit   = zeros(n_days)
    daily_SoC_end  = zeros(n_days)

    daily_charge     = Matrix{Float64}(undef, T_day, n_days)
    daily_discharge  = Matrix{Float64}(undef, T_day, n_days)
    daily_grid_sell  = Matrix{Float64}(undef, T_day, n_days)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*T_day + 1) : (d*T_day)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        params_day = BatteryParams(
            params.E_max,
            params.P_charge_max,
            params.P_discharge_max,
            params.eta_charge,
            params.eta_discharge,
            SoC_current,
            params.k_deg,
            params.Δt
        )

        println("\n=== Solving day $d (budgeted robust) ===")
        model = build_battery_model_budgeted(
            prices_day, solar_day, params_day;
            robust_price = true,
            Γ_price      = Γ_price,
            dev_frac     = dev_frac
        )

        optimize!(model)
        term = termination_status(model)
        println("Termination status (day $d): ", term)

        if term != MOI.OPTIMAL && term != MOI.LOCALLY_SOLVED
            @warn "Model for day $d did not reach OPTIMAL; results may be invalid."
        end

        daily_profit[d] = objective_value(model)

        charge    = value.(model[:charge])
        discharge = value.(model[:discharge])
        grid_sell = value.(model[:grid_sell])
        SoC       = value.(model[:SoC])

        daily_charge[:, d]    = charge
        daily_discharge[:, d] = discharge
        daily_grid_sell[:, d] = grid_sell

        daily_SoC_end[d] = SoC[end]
        SoC_current = SoC[end]

        println("Day $d profit: ", daily_profit[d])
        println("End-of-day SoC: ", SoC_current)
    end

    println("\n==============================")
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)
    
    println("Daily profits over 7 days (budgeted):")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    # ---------- Export results, with timestamps ----------
    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)
    df_results = DataFrame(
        day       = day_id,
        hour      = hours,
        charge    = round.(vec(daily_charge), digits=2),
        discharge = round.(vec(daily_discharge), digits=2),
        grid_sell = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results_budgeted.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)

    # Load timestamps
    df_solar = CSV.read(solar_path, DataFrame)
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")
    @assert nrow(df_solar) == 168
    
    df_results[!, :datetime_utc] = df_solar.datetime_utc
    
    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)
    
    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")
    
    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )
    
    out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps_budgeted.csv")
    CSV.write(out_path2, df_results)
    println("\nSaved WITH timestamps → ", out_path2)

    #############################
    # OPERATOR PRESCRIPTION EXPORT (JULIA)
    #############################
    
    # --- Battery constants (must match optimization) ---
    E_MAX = params.E_max
    ETA_CHARGE = params.eta_charge
    ETA_DISCHARGE = params.eta_discharge
    DT = params.Δt
    
    # ------------------------------
    # CLASSIFY ACTION PER HOUR
    # ------------------------------
    function classify_action(charge, discharge, grid_sell)
        if charge > 0.01
            return "CHARGE"
        elseif discharge > 0.01
            return "DISCHARGE"
        elseif grid_sell > 0.01
            return "SELL"
        else
            return "HOLD"
        end
    end
    
    df_results.action = [
        classify_action(df_results.charge[i],
                        df_results.discharge[i],
                        df_results.grid_sell[i])
        for i in 1:nrow(df_results)
    ]
    
    # ------------------------------
    # COMPUTE SOC TRAJECTORY
    # ------------------------------
    soc = zeros(nrow(df_results) + 1)
    soc[1] = params.SoC0
    
    for i in 1:nrow(df_results)
        soc[i+1] = soc[i] +
                   ETA_CHARGE * df_results.charge[i] * DT -
                   (1 / ETA_DISCHARGE) * df_results.discharge[i] * DT
    
        soc[i+1] = clamp(soc[i+1], 0.0, E_MAX)
    end

    df_results.soc_start = soc[1:end-1]
    df_results.soc_end   = soc[2:end]
    
    df_results.soc_pct_start = 100 .* df_results.soc_start ./ E_MAX
    df_results.soc_pct_end   = 100 .* df_results.soc_end   ./ E_MAX
    
    # ------------------------------
    # GROUP INTO CONTINUOUS ACTION BLOCKS
    # ------------------------------
    action_block = ones(Int, nrow(df_results))
    
    for i in 2:nrow(df_results)
        action_block[i] = df_results.action[i] == df_results.action[i-1] ?
                          action_block[i-1] : action_block[i-1] + 1
    end
    
    df_results.action_block = action_block
    
    summary = combine(groupby(df_results, :action_block),
        :datetime_ca => first => :start_time,
        :datetime_ca => last  => :end_time,
        :action      => first => :action,
        :charge      => mean  => :avg_charge,
        :discharge   => mean  => :avg_discharge,
        :soc_start   => first => :soc_start,
        :soc_end     => last  => :soc_end,
        :soc_pct_start => first => :soc_pct_start,
        :soc_pct_end   => last  => :soc_pct_end
    )
    
    # ------------------------------
    # ROUND FOR READABILITY
    # ------------------------------
    summary.avg_charge     = round.(summary.avg_charge, digits=2)
    summary.avg_discharge  = round.(summary.avg_discharge, digits=2)
    summary.soc_start      = round.(summary.soc_start, digits=2)
    summary.soc_end        = round.(summary.soc_end, digits=2)
    summary.soc_pct_start = round.(summary.soc_pct_start, digits=1)
    summary.soc_pct_end   = round.(summary.soc_pct_end, digits=1)
    
    # ------------------------------
    # GENERATE OPERATOR INSTRUCTIONS
    # ------------------------------
    summary.MW_instruction = [
        "From $(summary.start_time[i]) to $(summary.end_time[i]) → " *
        "$(summary.action[i]) at ~" *
        string(summary.action[i] == "CHARGE" ?
               summary.avg_charge[i] :
               summary.avg_discharge[i]) * " MW"
        for i in 1:nrow(summary)
    ]
    
    summary.SOC_instruction = [
        "From $(summary.start_time[i]) → SoC " *
        "$(summary.soc_pct_start[i])% → $(summary.soc_pct_end[i])% during " *
        "$(summary.action[i])"
        for i in 1:nrow(summary)
    ]
    
    # ------------------------------
    # EXPORT FINAL OPERATOR CSV
    # ------------------------------
    operator_path = joinpath(project_dir, "results", "budgeted_battery_operator_prescription.csv")
    CSV.write(operator_path, summary)
    
    println("\n✅ Operator prescription saved to:")
    println(operator_path)
end

main()

Reading CAISO LMP from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/sdge_2024_lmp.csv
Reading ORT solar forecasts from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/ghi_predictions_new_time.csv

=== Solving day 1 (budgeted robust) ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Max
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 97 rows, 122 columns and 265 nonzeros
Model fingerprint: 0x81c32a8b
Coefficient statistics:
  Matrix range     [9e-01, 1e+01]
  Objective range  [1e+00, 6e+01]
  Bounds range     [5e+00, 2e+01]
  RHS range        [1e-03, 3e+00]
Presolve removed 50 rows and 27 columns
Presolve time: 0.00s
Presolved: 47 rows, 95 columns, 187 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.4072584

# Box Uncertainty on Price

In [28]:
#############################
# Solar–Battery 7-Day Optimization
# BOX Robust Pricing (Worst-Case 20% Drop)
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64
    P_charge_max::Float64
    P_discharge_max::Float64
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64
    k_deg::Float64
    Δt::Float64
end

#############################
# BOX Robust Battery Model
#############################

function build_battery_model_box(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_price::Bool=false,
    dev_frac::Float64=0.20
)
    T = length(prices)
    @assert length(solar) == T "prices and solar must have same length"

    model = Model(Gurobi.Optimizer)

    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max)
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)
    @variable(model, 0 <= grid_sell[1:T])

    @constraint(model, SoC[0] == params.SoC0)

    for t in 1:T
        @constraint(model,
            SoC[t] == SoC[t-1] +
                      params.eta_charge          * charge[t]   * params.Δt -
                      (1 / params.eta_discharge) * discharge[t] * params.Δt
        )

        @constraint(model, charge[t] <= solar[t])
        @constraint(model, grid_sell[t] == solar[t] - charge[t] + discharge[t])
    end

    # -------------------------------
    # ✅ BOX ROBUST OBJECTIVE
    # -------------------------------
    if robust_price
        worst_prices = (1 .- dev_frac) .* prices

        @objective(model, Max,
            params.Δt * (
                sum(worst_prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    else
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    end

    return model
end

#############################
# Main script — BOX Robust
#############################

function main()
    project_dir = ".."

    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    println("Reading CAISO LMP from: ", lmp_path)
    df_lmp = CSV.read(lmp_path, DataFrame)

    println("Reading ORT solar forecasts from: ", solar_path)
    df_solar = CSV.read(solar_path, DataFrame)

    N = 168
    @assert nrow(df_solar) == N

    @assert nrow(df_lmp) >= N
    shift_hours = 48
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours

    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])
    @assert length(prices_all) == N

    I_ref        = 1000.0
    P_solar_max  = 5.0
    solar_irr = df_solar.prediction
    solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
    solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)
    solar_all = Vector{Float64}(solar_mw)

    params = BatteryParams(
        E_max           = 20.0,
        P_charge_max    = 5.0,
        P_discharge_max = 5.0,
        eta_charge      = 0.95,
        eta_discharge   = 0.95,
        SoC0            = 0.0,
        k_deg           = 5.0,
        Δt              = 1.0
    )

    # ✅ BOX Robustness knob
    dev_frac = 0.20   # 20% independent worst-case price drop

    # ---------- Day-by-day optimization ----------
    T_day = 24
    n_days = N ÷ T_day
    @assert n_days == 7

    daily_profit   = zeros(n_days)
    daily_SoC_end  = zeros(n_days)

    daily_charge     = Matrix{Float64}(undef, T_day, n_days)
    daily_discharge  = Matrix{Float64}(undef, T_day, n_days)
    daily_grid_sell  = Matrix{Float64}(undef, T_day, n_days)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*T_day + 1):(d*T_day)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        params_day = BatteryParams(
            params.E_max,
            params.P_charge_max,
            params.P_discharge_max,
            params.eta_charge,
            params.eta_discharge,
            SoC_current,
            params.k_deg,
            params.Δt
        )

        println("\n=== Solving day $d (BOX robust) ===")

        model = build_battery_model_box(
            prices_day, solar_day, params_day;
            robust_price = true,
            dev_frac     = dev_frac
        )

        optimize!(model)
        term = termination_status(model)
        println("Termination status (day $d): ", term)

        daily_profit[d] = objective_value(model)

        charge    = value.(model[:charge])
        discharge = value.(model[:discharge])
        grid_sell = value.(model[:grid_sell])
        SoC       = value.(model[:SoC])

        daily_charge[:, d]    = charge
        daily_discharge[:, d] = discharge
        daily_grid_sell[:, d] = grid_sell

        daily_SoC_end[d] = SoC[end]
        SoC_current      = SoC[end]

        println("Day $d profit: ", daily_profit[d])
        println("End-of-day SoC: ", SoC_current)
    end

    println("\n==============================")
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)

    println("Daily profits over 7 days (BOX robust):")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    # ---------- Export results ----------
    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)

    df_results = DataFrame(
        day       = day_id,
        hour      = hours,
        charge    = round.(vec(daily_charge), digits=2),
        discharge = round.(vec(daily_discharge), digits=2),
        grid_sell = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results_box.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)

    # ---------- Add timestamps ----------
    df_solar = CSV.read(solar_path, DataFrame)
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")

    df_results[!, :datetime_utc] = df_solar.datetime_utc

    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)

    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")

    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )

    out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps_box.csv")
    CSV.write(out_path2, df_results)

    println("\nSaved WITH timestamps → ", out_path2)
end

main()

Reading CAISO LMP from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/sdge_2024_lmp.csv
Reading ORT solar forecasts from: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/ghi_predictions_new_time.csv

=== Solving day 1 (BOX robust) ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Max
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 73 rows, 97 columns and 193 nonzeros
Model fingerprint: 0x4e724ee8
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [5e+00, 5e+01]
  Bounds range     [5e+00, 2e+01]
  RHS range        [1e-03, 3e+00]
Presolve removed 50 rows and 29 columns
Presolve time: 0.00s
Presolved: 23 rows, 68 columns, 90 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.6518329e+03   

# Charge Day, Discharge at Night

In [7]:
#############################
# RULE-BASED BATTERY BASELINE
# (Standalone Block)
#############################
using CSV, DataFrames

project_dir = ".."

# Load prices
df_lmp = CSV.read(joinpath(project_dir, "data", "sdge_2024_lmp.csv"), DataFrame)

# Load solar
df_solar = CSV.read(joinpath(project_dir, "data", "ghi_predictions_new_time.csv"), DataFrame)

# --- Extract last 168 hours (match your original logic) ---
N = 168
shift_hours = 48

start_idx = nrow(df_lmp) - N - shift_hours + 1
end_idx   = nrow(df_lmp) - shift_hours

prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])

# --- Convert solar irradiance to MW (same as your main script) ---
I_ref        = 1000.0
P_solar_max  = 5.0

solar_irr = df_solar.prediction
solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)

solar_all = Vector{Float64}(solar_mw)

@assert length(prices_all) == 168
@assert length(solar_all)  == 168


using Statistics

# ---------------------------
# Battery parameters struct
# ---------------------------
Base.@kwdef struct BatteryParams
    E_max::Float64
    P_charge_max::Float64
    P_discharge_max::Float64
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64
    k_deg::Float64
    Δt::Float64
end

# ---------------------------
# ✅ DEFINE PARAMS LOCALLY
# ---------------------------
params = BatteryParams(
    E_max           = 20.0,   # MWh
    P_charge_max    = 5.0,    # MW
    P_discharge_max = 5.0,    # MW
    eta_charge      = 0.95,
    eta_discharge   = 0.95,
    SoC0            = 0.0,    # initial SoC
    k_deg           = 5.0,    # $/MWh discharged
    Δt              = 1.0     # 1-hour steps
)

# ---------------------------------------------------
# Rule-Based Battery Dispatch (Heuristic Controller)
# ---------------------------------------------------
function baseline_battery_rule_based(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams
)
    T = length(prices)
    @assert length(solar) == T

    charge     = zeros(T)
    discharge  = zeros(T)
    grid_sell  = zeros(T)
    SoC        = zeros(T+1)
    SoC[1] = params.SoC0

    p_low  = quantile(prices, 0.10)
    p_high = quantile(prices, 0.90)

    total_profit = 0.0

    for t in 1:T
        ch_t  = 0.0
        dis_t = 0.0

        # ---- Rule 1: Charge when price is cheap and solar exists ----
        if solar[t] > 0.01 && prices[t] <= p_low
            ch_t = min(
                params.P_charge_max,
                solar[t],
                (params.E_max - SoC[t]) / params.Δt
            )

        # ---- Rule 2: Discharge when price is high ----
        elseif prices[t] ≥ p_high
            dis_t = max(0.0, min(
                        params.P_discharge_max,
                        SoC[t] / params.Δt
                                            ))

        end

        charge[t]    = ch_t
        discharge[t] = dis_t

        # ---- SoC update ----
        SoC[t+1] = SoC[t] +
                   params.eta_charge * ch_t * params.Δt -
                   (1 / params.eta_discharge) * dis_t * params.Δt

        # ---- Grid export (solar + discharge) ----
        grid_sell[t] = solar[t] - ch_t + dis_t
        # ---- Hourly profit ----
        total_profit += params.Δt * (
            prices[t] * grid_sell[t] -
            params.k_deg * dis_t
        )
    end

    return total_profit, charge, discharge, grid_sell, SoC
end


# ---------------------------------------------------
# ✅ 7-DAY RULE-BASED SIMULATION LOOP (NO main)
# ---------------------------------------------------

T_day  = 24
N      = length(prices_all)     # assumes 168
n_days = N ÷ T_day

daily_profit_rule   = zeros(n_days)
daily_SoC_end_rule  = zeros(n_days)

daily_charge_rule    = Matrix{Float64}(undef, T_day, n_days)
daily_discharge_rule = Matrix{Float64}(undef, T_day, n_days)
daily_grid_sell_rule = Matrix{Float64}(undef, T_day, n_days)

SoC_current_rule = params.SoC0

for d in 1:n_days
    idx = ((d-1)*T_day + 1) : (d*T_day)

    prices_day = prices_all[idx]
    solar_day  = solar_all[idx]

    params_day = BatteryParams(
        params.E_max,
        params.P_charge_max,
        params.P_discharge_max,
        params.eta_charge,
        params.eta_discharge,
        SoC_current_rule,
        params.k_deg,
        params.Δt
    )

    profit_rule, ch_rule, dis_rule, sell_rule, soc_rule =
        baseline_battery_rule_based(prices_day, solar_day, params_day)

    daily_profit_rule[d] = profit_rule
    daily_charge_rule[:, d]    = ch_rule
    daily_discharge_rule[:, d] = dis_rule
    daily_grid_sell_rule[:, d] = sell_rule

    daily_SoC_end_rule[d] = soc_rule[end]
    SoC_current_rule = soc_rule[end]

    println("Rule-Based Battery — Day $d Profit: ",
        round(profit_rule, digits=2))
end

println("\n==============================")
println("RULE-BASED BATTERY BASELINE")
println("Daily profits:")
println(round.(daily_profit_rule, digits=2))
println("Total 7-day profit: ",
    round(sum(daily_profit_rule), digits=2))
println("End-of-day SoC:")
println(round.(daily_SoC_end_rule, digits=2))

using Dates

using Dates

#############################################
# FORMAT RULE-BASED RESULTS LIKE BUDGETED
#############################################

hours  = repeat(1:T_day, n_days)
day_id = vcat([fill(d, T_day) for d in 1:n_days]...)

df_results = DataFrame(
    day       = day_id,
    hour      = hours,
    charge    = round.(vec(daily_charge_rule), digits=2),
    discharge = round.(vec(daily_discharge_rule), digits=2),
    grid_sell = round.(vec(daily_grid_sell_rule), digits=2),
)

# --------------------------
# Save WITHOUT timestamps
# --------------------------
out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results_rule_based.csv")
CSV.write(out_path, df_results)
println("\nSaved hourly decisions to: ", out_path)

# --------------------------
# ✅ Load & Attach Timestamps
# --------------------------
solar_path = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")
df_solar   = CSV.read(solar_path, DataFrame)

fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")

@assert nrow(df_solar) == 168

df_results[!, :datetime_utc] = df_solar.datetime_utc

pacific = tz"America/Los_Angeles"
df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)

df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")

# --------------------------
# ✅ Final Column Layout
# --------------------------
select!(df_results,
    :datetime_utc_str => :datetime_utc,
    :datetime_ca_str  => :datetime_ca,
    :charge,
    :discharge,
    :grid_sell
)

# --------------------------
# ✅ Save WITH timestamps
# --------------------------
out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps_rule_based.csv")
CSV.write(out_path2, df_results)

#############################
# RULE-BASED BATTERY BASELINE
# (Standalone Block)
#############################
using CSV, DataFrames

project_dir = ".."

# Load prices
df_lmp = CSV.read(joinpath(project_dir, "data", "sdge_2024_lmp.csv"), DataFrame)

# Load solar
df_solar = CSV.read(joinpath(project_dir, "data", "ghi_predictions_new_time.csv"), DataFrame)

# --- Extract last 168 hours (match your original logic) ---
N = 168
shift_hours = 48

start_idx = nrow(df_lmp) - N - shift_hours + 1
end_idx   = nrow(df_lmp) - shift_hours

prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])

# --- Convert solar irradiance to MW (same as your main script) ---
I_ref        = 1000.0
P_solar_max  = 5.0

solar_irr = df_solar.prediction
solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)

solar_all = Vector{Float64}(solar_mw)

@assert length(prices_all) == 168
@assert length(solar_all)  == 168


using Statistics

# ---------------------------
# Battery parameters struct
# ---------------------------
Base.@kwdef struct BatteryParams
    E_max::Float64
    P_charge_max::Float64
    P_discharge_max::Float64
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64
    k_deg::Float64
    Δt::Float64
end

# ---------------------------
# ✅ DEFINE PARAMS LOCALLY
# ---------------------------
params = BatteryParams(
    E_max           = 20.0,   # MWh
    P_charge_max    = 5.0,    # MW
    P_discharge_max = 5.0,    # MW
    eta_charge      = 0.95,
    eta_discharge   = 0.95,
    SoC0            = 0.0,    # initial SoC
    k_deg           = 5.0,    # $/MWh discharged
    Δt              = 1.0     # 1-hour steps
)

# ---------------------------------------------------
# Rule-Based Battery Dispatch (Heuristic Controller)
# ---------------------------------------------------
function baseline_battery_rule_based(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams
)
    T = length(prices)
    @assert length(solar) == T

    charge     = zeros(T)
    discharge  = zeros(T)
    grid_sell  = zeros(T)
    SoC        = zeros(T+1)
    SoC[1] = params.SoC0

    p_low  = quantile(prices, 0.10)
    p_high = quantile(prices, 0.90)

    total_profit = 0.0

    for t in 1:T
        ch_t  = 0.0
        dis_t = 0.0

        # ---- Rule 1: Charge when price is cheap and solar exists ----
        if solar[t] > 0.01 && prices[t] <= p_low
            ch_t = min(
                params.P_charge_max,
                solar[t],
                (params.E_max - SoC[t]) / params.Δt
            )

        # ---- Rule 2: Discharge when price is high ----
        elseif prices[t] ≥ p_high
            dis_t = max(0.0, min(
                        params.P_discharge_max,
                        SoC[t] / params.Δt
                                            ))

        end

        charge[t]    = ch_t
        discharge[t] = dis_t

        # ---- SoC update ----
        SoC[t+1] = SoC[t] +
                   params.eta_charge * ch_t * params.Δt -
                   (1 / params.eta_discharge) * dis_t * params.Δt

        # ---- Grid export (solar + discharge) ----
        grid_sell[t] = solar[t] - ch_t + dis_t
        # ---- Hourly profit ----
        total_profit += params.Δt * (
            prices[t] * grid_sell[t] -
            params.k_deg * dis_t
        )
    end

    return total_profit, charge, discharge, grid_sell, SoC
end


# ---------------------------------------------------
# ✅ 7-DAY RULE-BASED SIMULATION LOOP (NO main)
# ---------------------------------------------------

T_day  = 24
N      = length(prices_all)     # assumes 168
n_days = N ÷ T_day

daily_profit_rule   = zeros(n_days)
daily_SoC_end_rule  = zeros(n_days)

daily_charge_rule    = Matrix{Float64}(undef, T_day, n_days)
daily_discharge_rule = Matrix{Float64}(undef, T_day, n_days)
daily_grid_sell_rule = Matrix{Float64}(undef, T_day, n_days)

SoC_current_rule = params.SoC0

for d in 1:n_days
    idx = ((d-1)*T_day + 1) : (d*T_day)

    prices_day = prices_all[idx]
    solar_day  = solar_all[idx]

    params_day = BatteryParams(
        params.E_max,
        params.P_charge_max,
        params.P_discharge_max,
        params.eta_charge,
        params.eta_discharge,
        SoC_current_rule,
        params.k_deg,
        params.Δt
    )

    profit_rule, ch_rule, dis_rule, sell_rule, soc_rule =
        baseline_battery_rule_based(prices_day, solar_day, params_day)

    daily_profit_rule[d] = profit_rule
    daily_charge_rule[:, d]    = ch_rule
    daily_discharge_rule[:, d] = dis_rule
    daily_grid_sell_rule[:, d] = sell_rule

    daily_SoC_end_rule[d] = soc_rule[end]
    SoC_current_rule = soc_rule[end]

    println("Rule-Based Battery — Day $d Profit: ",
        round(profit_rule, digits=2))
end

println("\n==============================")
println("RULE-BASED BATTERY BASELINE")
println("Daily profits:")
println(round.(daily_profit_rule, digits=2))
println("Total 7-day profit: ",
    round(sum(daily_profit_rule), digits=2))
println("End-of-day SoC:")
println(round.(daily_SoC_end_rule, digits=2))

using Dates

using Dates

#############################################
# FORMAT RULE-BASED RESULTS LIKE BUDGETED
#############################################

hours  = repeat(1:T_day, n_days)
day_id = vcat([fill(d, T_day) for d in 1:n_days]...)

df_results = DataFrame(
    day       = day_id,
    hour      = hours,
    charge    = round.(vec(daily_charge_rule), digits=2),
    discharge = round.(vec(daily_discharge_rule), digits=2),
    grid_sell = round.(vec(daily_grid_sell_rule), digits=2),
)

# --------------------------
# Save WITHOUT timestamps
# --------------------------
out_path = joinpath(project_dir, "results", "battery_dispatch_7days_results_rule_based.csv")
CSV.write(out_path, df_results)
println("\nSaved hourly decisions to: ", out_path)

# --------------------------
# ✅ Load & Attach Timestamps
# --------------------------
solar_path = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")
df_solar   = CSV.read(solar_path, DataFrame)

fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")

@assert nrow(df_solar) == 168

df_results[!, :datetime_utc] = df_solar.datetime_utc

pacific = tz"America/Los_Angeles"
df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)

df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")

# --------------------------
# ✅ Final Column Layout
# --------------------------
select!(df_results,
    :datetime_utc_str => :datetime_utc,
    :datetime_ca_str  => :datetime_ca,
    :charge,
    :discharge,
    :grid_sell
)

# --------------------------
# ✅ Save WITH timestamps
# --------------------------
out_path2 = joinpath(project_dir, "results", "battery_dispatch_with_timestamps_rule_based.csv")
CSV.write(out_path2, df_results)

   #############################
    # OPERATOR PRESCRIPTION EXPORT (JULIA)
    #############################
    
    # --- Battery constants (must match optimization) ---
    E_MAX = params.E_max
    ETA_CHARGE = params.eta_charge
    ETA_DISCHARGE = params.eta_discharge
    DT = params.Δt
    
    # ------------------------------
    # CLASSIFY ACTION PER HOUR
    # ------------------------------
    function classify_action(charge, discharge, grid_sell)
        if charge > 0.01
            return "CHARGE"
        elseif discharge > 0.01
            return "DISCHARGE"
        elseif grid_sell > 0.01
            return "SELL"
        else
            return "HOLD"
        end
    end
    
    df_results.action = [
        classify_action(df_results.charge[i],
                        df_results.discharge[i],
                        df_results.grid_sell[i])
        for i in 1:nrow(df_results)
    ]
    
    # ------------------------------
    # COMPUTE SOC TRAJECTORY
    # ------------------------------
    soc = zeros(nrow(df_results) + 1)
    soc[1] = params.SoC0
    
    for i in 1:nrow(df_results)
        soc[i+1] = soc[i] +
                   ETA_CHARGE * df_results.charge[i] * DT -
                   (1 / ETA_DISCHARGE) * df_results.discharge[i] * DT
    
        soc[i+1] = clamp(soc[i+1], 0.0, E_MAX)
    end

    df_results.soc_start = soc[1:end-1]
    df_results.soc_end   = soc[2:end]
    
    df_results.soc_pct_start = 100 .* df_results.soc_start ./ E_MAX
    df_results.soc_pct_end   = 100 .* df_results.soc_end   ./ E_MAX
    
    # ------------------------------
    # GROUP INTO CONTINUOUS ACTION BLOCKS
    # ------------------------------
    action_block = ones(Int, nrow(df_results))
    
    for i in 2:nrow(df_results)
        action_block[i] = df_results.action[i] == df_results.action[i-1] ?
                          action_block[i-1] : action_block[i-1] + 1
    end
    
    df_results.action_block = action_block
    
    summary = combine(groupby(df_results, :action_block),
        :datetime_ca => first => :start_time,
        :datetime_ca => last  => :end_time,
        :action      => first => :action,
        :charge      => mean  => :avg_charge,
        :discharge   => mean  => :avg_discharge,
        :soc_start   => first => :soc_start,
        :soc_end     => last  => :soc_end,
        :soc_pct_start => first => :soc_pct_start,
        :soc_pct_end   => last  => :soc_pct_end
    )
    
    # ------------------------------
    # ROUND FOR READABILITY
    # ------------------------------
    summary.avg_charge     = round.(summary.avg_charge, digits=2)
    summary.avg_discharge  = round.(summary.avg_discharge, digits=2)
    summary.soc_start      = round.(summary.soc_start, digits=2)
    summary.soc_end        = round.(summary.soc_end, digits=2)
    summary.soc_pct_start = round.(summary.soc_pct_start, digits=1)
    summary.soc_pct_end   = round.(summary.soc_pct_end, digits=1)
    
    # ------------------------------
    # GENERATE OPERATOR INSTRUCTIONS
    # ------------------------------
    summary.MW_instruction = [
        "From $(summary.start_time[i]) to $(summary.end_time[i]) → " *
        "$(summary.action[i]) at ~" *
        string(summary.action[i] == "CHARGE" ?
               summary.avg_charge[i] :
               summary.avg_discharge[i]) * " MW"
        for i in 1:nrow(summary)
    ]
    
    summary.SOC_instruction = [
        "From $(summary.start_time[i]) → SoC " *
        "$(summary.soc_pct_start[i])% → $(summary.soc_pct_end[i])% during " *
        "$(summary.action[i])"
        for i in 1:nrow(summary)
    ]
    
    # ------------------------------
    # EXPORT FINAL OPERATOR CSV
    # ------------------------------
    operator_path = joinpath(project_dir, "results", "heuristic_battery_operator_prescription.csv")
    CSV.write(operator_path, summary)
    
    println("\n✅ Operator prescription saved to:")
    println(operator_path)


println("\nSaved WITH timestamps → ", out_path2)

Rule-Based Battery — Day 1 Profit: 629.53
Rule-Based Battery — Day 2 Profit: 328.99
Rule-Based Battery — Day 3 Profit: 554.04
Rule-Based Battery — Day 4 Profit: 554.35
Rule-Based Battery — Day 5 Profit: 444.01
Rule-Based Battery — Day 6 Profit: 428.23
Rule-Based Battery — Day 7 Profit: 268.44

RULE-BASED BATTERY BASELINE
Daily profits:
[629.53, 328.99, 554.04, 554.35, 444.01, 428.23, 268.44]
Total 7-day profit: 3207.59
End-of-day SoC:
[-0.03, 4.33, -0.03, -0.25, -0.26, -0.04, -0.25]

Saved hourly decisions to: /Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/battery_dispatch_7days_results_rule_based.csv
Rule-Based Battery — Day 1 Profit: 629.53
Rule-Based Battery — Day 2 Profit: 328.99
Rule-Based Battery — Day 3 Profit: 554.04
Rule-Based Battery — Day 4 Profit: 554.35
Rule-Based Battery — Day 5 Profit: 444.01
Rule-Based Battery — Day 6 Profit: 428.23
Rule-Based Battery — Day 7 Profit: 268.44

RULE-BASED BATTERY BASELINE
Daily profits:
[629.53, 328.99, 554.04, 554.35, 4

# L-0 Norm on Solar

In [3]:
#############################
# Solar–Battery 7-Day Optimization
# Deterministic (point forecast) version
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates



# Choose a solver you have installed:
using Gurobi
 # LP solver; fine since robust_price=false in this script

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64          # MWh - energy capacity
    P_charge_max::Float64   # MW  - max charging power
    P_discharge_max::Float64 # MW  - max discharging power
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64           # MWh - initial state of charge
    k_deg::Float64          # $/MWh discharged (degradation cost)
    Δt::Float64             # hours per time step (usually 1.0)
end



function build_battery_model(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_price::Bool=false,
    Γ_price::Float64=0.0,
    robust_solar::Bool=false,      # ✅ ADD THIS
    γ_solar::Float64=0.0           # ✅ ADD THIS
)

    T = length(prices)
    @assert length(solar) == T "prices and solar must have same length"

    # Pick solver here
    model = Model(Gurobi.Optimizer)

    # Decision variables
    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)    # MW
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max) # MW
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)           # MWh
    @variable(model, 0 <= grid_sell[1:T])                                   # MWh

    # If robust_price is turned on, add variable & SOC constraint
    if robust_price
        @variable(model, z >= 0)  # upper bound on ||grid_sell||_2
        @constraint(model, [z; grid_sell] in MOI.SecondOrderCone())
    end

    # Initial SoC
    @constraint(model, SoC[0] == params.SoC0)

    # Battery dynamics + solar coupling + export balance
    for t in 1:T
        # State-of-charge evolution
        @constraint(model,
            SoC[t] == SoC[t-1] +
                      params.eta_charge          * charge[t]   * params.Δt -
                      (1 / params.eta_discharge) * discharge[t] * params.Δt
        )

        # Charge can only come from solar (no grid charging)
        if robust_solar
            solar_min = (1 - γ_solar) * solar[t]

            @constraint(model, charge[t] <= solar_min)
            @constraint(model,
                        grid_sell[t] == solar_min - charge[t] + discharge[t]
                        )
            else
            @constraint(model, charge[t] <= solar[t])
            @constraint(model,
                        grid_sell[t] == solar[t] - charge[t] + discharge[t]
                        )
            end

        # (export[t] >= 0 enforced by variable bounds)
    end

    # Objective: profit = sum_t (price_t * export_t - k_deg * discharge_t) * Δt
    if robust_price
        # Robust price revenue: sum(π̂_t e_t) - Γ * z
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                Γ_price * model[:z] -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    else
        @objective(model, Max,
            params.Δt * (
                sum(prices[t] * grid_sell[t] for t in 1:T) -
                params.k_deg * sum(discharge[t] for t in 1:T)
            )
        )
    end

    return model
end

#############################
# Main script
#############################

function main()
    # ---------- File paths ----------
    project_dir = ".."

    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    println("Reading CAISO LMP from: ", lmp_path)
    df_lmp = CSV.read(lmp_path, DataFrame)

    println("Reading ORT solar forecasts from: ", solar_path)
    df_solar = CSV.read(solar_path, DataFrame)

    # ---------- Extract last 7 days (168 hours) ----------
    N = 168  # 7 days * 24 hours
    @assert nrow(df_solar) == N 

    # CAISO: take last 168 LMP values as "last week of year"
    @assert nrow(df_lmp) >= N "Not enough LMP rows to extract last 168 hours."
    # Shift back LMP window by 1 day (24 hours)
    shift_hours = 48  # because solar ends 2 days earlier than LMP file
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours
    
    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])
    
    @assert length(prices_all) == 168

    # ORT irradiance predictions (assumed already in MWh per hour or convertible)
    I_ref        = 1000.0          # W/m²
    P_solar_max  = 5.0             # MW, "decent sized" PV for a 5 MW / 20 MWh battery
    
    solar_irr = df_solar.prediction              # W/m²
    solar_mw  = (solar_irr ./ I_ref) .* P_solar_max
    solar_mw  = clamp.(solar_mw, 0.0, P_solar_max)
    
    solar_all = Vector{Float64}(solar_mw)  # MWh per hour since Δt = 1
    @assert length(solar_all) == N "Solar forecast vector must be length 168."

    # ---------- Battery parameters ----------
    params = BatteryParams(
        E_max           = 20.0,   # MWh
        P_charge_max    = 5.0,    # MW
        P_discharge_max = 5.0,    # MW
        eta_charge      = 0.95,
        eta_discharge   = 0.95,
        SoC0            = 0,   # initial SoC at start of 7-day horizon
        k_deg           = 5.0,    # $ per MWh discharged
        Δt              = 1.0    # 1-hour time steps
    )

    # ---------- Day-by-day optimization (7 days) ----------
    T_day = 24
    n_days = N ÷ T_day
    @assert n_days == 7 "Expected 7 days (168 / 24)."

    daily_profit   = zeros(n_days)
    daily_SoC_end  = zeros(n_days)

    # Store hourly decisions per day (rows = hour, cols = day)
    daily_charge     = Matrix{Float64}(undef, T_day, n_days)
    daily_discharge  = Matrix{Float64}(undef, T_day, n_days)
    daily_grid_sell     = Matrix{Float64}(undef, T_day, n_days)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*T_day + 1) : (d*T_day)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        # Use carryover SoC from previous day as SoC0
        params_day = BatteryParams(
            params.E_max,
            params.P_charge_max,
            params.P_discharge_max,
            params.eta_charge,
            params.eta_discharge,
            SoC_current,
            params.k_deg,
            params.Δt
        )

        println("\n=== Solving day $d ===")
        model = build_battery_model(prices_day, solar_day, params_day;
                                    robust_solar = true,      
                                    γ_solar = 0.20)

        optimize!(model)
        term = termination_status(model)
        println("Termination status (day $d): ", term)

        if term != MOI.OPTIMAL && term != MOI.LOCALLY_SOLVED
            @warn "Model for day $d did not reach OPTIMAL; results may be invalid."
        end

        # Extract results
        daily_profit[d] = objective_value(model)

        charge    = value.(model[:charge])
        discharge = value.(model[:discharge])
        grid_sell    = value.(model[:grid_sell])
        SoC       = value.(model[:SoC])

        daily_charge[:, d]    = charge
        daily_discharge[:, d] = discharge
        daily_grid_sell[:, d]    = grid_sell

        daily_SoC_end[d] = SoC[end]
        SoC_current = SoC[end]  # carry SoC to next starting day

        println("Day $d profit: ", daily_profit[d])
        println("End-of-day SoC: ", SoC_current)
    end

    println("\n==============================")
    # Round daily and total profit
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)
    
    println("==============================")
    println("Daily profits over 7 days:")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    # If you want to save results to CSV for plotting/reporting:
    # construct a DataFrame of hourly decisions
    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)
    df_results = DataFrame(
        day   = day_id,
        hour  = hours,
        charge     = round.(vec(daily_charge), digits=2),
        discharge  = round.(vec(daily_discharge), digits=2),
        grid_sell  = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_solar_uncertainty_7days_results.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)


   

    # Load timestamps
    solar_path = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")
    df_solar = CSV.read(solar_path, DataFrame)
    
    # Fix formatting: replace space with "T"
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    
    # Now parse correctly as ZonedDateTime
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")
    
    @assert nrow(df_solar) == 168
    
    # Attach timestamps
    df_results[!, :datetime_utc] = df_solar.datetime_utc
    
    # Convert to California time
    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)
    
    # Reorder
    # Pretty string formats
    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")
    
    # Reorder and KEEP ONLY formatted versions
    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )
    
    # Save
    out_path2 = joinpath(project_dir, "results", "battery_dispatch_solar_uncertainty_with_timestamps.csv")
    CSV.write(out_path2, df_results)
    println("\nSaved WITH timestamps → ", out_path2)


    # # Create a directory for plots
    # plots_dir = joinpath(project_dir, "battery_action_plots")
    # isdir(plots_dir) || mkdir(plots_dir)

    # action_label(x) = x == 1 ? "Charge" : x == 2 ? "Hold" : "Sell"

    # for d in 1:n_days
    #     hours = 1:T_day
    #     charge_d = daily_charge[:, d]
    #     sell_d   = daily_grid_sell[:, d]
    
    #     action = fill(2, T_day)  # default: Hold (2)
    
    #     for t in 1:T_day
    #         if charge_d[t] > 0.01
    #             action[t] = 1    # Charge
    #         elseif sell_d[t] > 0.01
    #             action[t] = 3    # Sell
    #         end
    #     end
    
    #     # Convert to strings for categorical y-axis
    #     action_str = [action_label(a) for a in action]
    
    #     p = plot(
    #         hours,
    #         action_str,
    #         seriestype = :scatter,
    #         markersize = 6,
    #         ylabel = "Action",
    #         xlabel = "Hour of Day",
    #         title = "Battery Action Schedule — Day $d",
    #         legend = false,
    #         yticks = (1:3, ["Charge","Hold","Sell"])
    #     )
    
    #     savefig(p, joinpath(plots_dir, "day$(d)_actions.png"))
    #     println("Saved plot for day $d.")
    # end

end

# Run main if script is executed directly
main()

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file:///Users/jjmurdock/Desktop/MIT/15.095-Machine_Learning_Opts/Project/In[3]#348:10\[90mIn[3]:348:10[0;0m]8;;\
    #     println("Saved plot for day $d.")
    # end[48;2;120;70;70m[0;0m
[90m#        └ ── [0;0m[91mExpected `end`[0;0m

# L-1 Norm on Solar

In [30]:
#############################
# Solar–Battery 7-Day Optimization
# ROBUST SOLAR (L1 / Bertsimas–Sim Budgeted)
#############################

using CSV
using DataFrames
using JuMP
using Gurobi
using LinearAlgebra
using MathOptInterface
const MOI = MathOptInterface
using Plots
using TimeZones
using Dates

#############################
# Battery model definition
#############################

Base.@kwdef struct BatteryParams
    E_max::Float64
    P_charge_max::Float64
    P_discharge_max::Float64
    eta_charge::Float64
    eta_discharge::Float64
    SoC0::Float64
    k_deg::Float64
    Δt::Float64
end

#############################
# ✅ CORRECT L1 ROBUST SOLAR MODEL
#############################

function build_battery_model_solar_L1(
    prices::Vector{Float64},
    solar::Vector{Float64},
    params::BatteryParams;
    robust_solar::Bool = false,
    Γ_solar::Float64 = 0.0,
    dev_frac::Float64 = 0.20
)

    T = length(prices)
    @assert length(solar) == T

    model = Model(Gurobi.Optimizer)

    @variable(model, 0 <= charge[1:T]      <= params.P_charge_max)
    @variable(model, 0 <= discharge[1:T]   <= params.P_discharge_max)
    @variable(model, 0 <= SoC[0:T]         <= params.E_max)
    @variable(model, 0 <= grid_sell[1:T])

    @constraint(model, SoC[0] == params.SoC0)

    # ✅ Budgeted solar deviation
    if robust_solar
        δ = dev_frac .* solar
        @variable(model, 0 <= ξ_s[1:T] <= 1)
        @constraint(model, sum(ξ_s) <= Γ_solar)
    end

    for t in 1:T
        @constraint(model,
            SoC[t] == SoC[t-1] +
                     params.eta_charge * charge[t] * params.Δt -
                     (1 / params.eta_discharge) * discharge[t] * params.Δt
        )

        if robust_solar
            @constraint(model,
                charge[t] <= solar[t] - δ[t] * ξ_s[t]
            )

            @constraint(model,
                grid_sell[t] == solar[t] - δ[t] * ξ_s[t] - charge[t] + discharge[t]
            )
        else
            @constraint(model, charge[t] <= solar[t])
            @constraint(model,
                grid_sell[t] == solar[t] - charge[t] + discharge[t]
            )
        end
    end

    @objective(model, Max,
        params.Δt * (
            sum(prices[t] * grid_sell[t] for t in 1:T)
            - params.k_deg * sum(discharge[t] for t in 1:T)
        )
    )

    return model
end

#############################
# Main Script — Robust Solar L1
#############################

function main()

    project_dir = ".."
    lmp_path    = joinpath(project_dir, "data", "sdge_2024_lmp.csv")
    solar_path  = joinpath(project_dir, "data", "ghi_predictions_new_time.csv")

    df_lmp   = CSV.read(lmp_path, DataFrame)
    df_solar = CSV.read(solar_path, DataFrame)

    N = 168
    shift_hours = 48
    start_idx = nrow(df_lmp) - N - shift_hours + 1
    end_idx   = nrow(df_lmp) - shift_hours

    prices_all = Vector{Float64}(df_lmp.LMP[start_idx:end_idx])

    I_ref = 1000.0
    P_solar_max = 5.0

    solar_irr = df_solar.prediction
    solar_mw  = clamp.((solar_irr ./ I_ref) .* P_solar_max, 0.0, P_solar_max)
    solar_all = Vector{Float64}(solar_mw)

    params = BatteryParams(
        E_max = 20.0,
        P_charge_max = 5.0,
        P_discharge_max = 5.0,
        eta_charge = 0.95,
        eta_discharge = 0.95,
        SoC0 = 0.0,
        k_deg = 5.0,
        Δt = 1.0
    )

    # ✅ TRUE L1 BUDGET PARAMETERS
    Γ_solar  = 5.0      # only 5 bad solar hours per day
    dev_frac = 0.20    # 20% max underproduction per bad hour

    T_day = 24
    n_days = 7

    daily_profit  = zeros(n_days)
    daily_SoC_end = zeros(n_days)

    daily_charge    = Matrix{Float64}(undef, 24, 7)
    daily_discharge = Matrix{Float64}(undef, 24, 7)
    daily_grid_sell = Matrix{Float64}(undef, 24, 7)

    SoC_current = params.SoC0

    for d in 1:n_days
        idx = ((d-1)*24 + 1):(d*24)

        prices_day = prices_all[idx]
        solar_day  = solar_all[idx]

        params_day = BatteryParams(
            params.E_max, params.P_charge_max, params.P_discharge_max,
            params.eta_charge, params.eta_discharge, SoC_current,
            params.k_deg, params.Δt
        )

        println("\n=== Solving day $d (TRUE L1 Robust Solar) ===")

        model = build_battery_model_solar_L1(
            prices_day, solar_day, params_day;
            robust_solar = true,
            Γ_solar = Γ_solar,
            dev_frac = dev_frac
        )

        optimize!(model)

        daily_profit[d] = objective_value(model)

        daily_charge[:,d]    = value.(model[:charge])
        daily_discharge[:,d] = value.(model[:discharge])
        daily_grid_sell[:,d] = value.(model[:grid_sell])

        SoC_current = value.(model[:SoC])[end]
        daily_SoC_end[d] = SoC_current
    end

    ##################################
    # ✅ OUTPUT + EXPORT BLOCK
    ##################################

    println("==============================")
    daily_profit_rounded = round.(daily_profit, digits=2)
    total_profit = round(sum(daily_profit), digits=2)

    println("Daily profits over 7 days:")
    println(daily_profit_rounded)
    println("Total profit over 7 days: ", total_profit)
    println("End-of-day SoC each day:")
    println(daily_SoC_end)

    hours = repeat(1:T_day, n_days)
    day_id = vcat([fill(d, T_day) for d in 1:n_days]...)

    df_results = DataFrame(
        day   = day_id,
        hour  = hours,
        charge     = round.(vec(daily_charge), digits=2),
        discharge  = round.(vec(daily_discharge), digits=2),
        grid_sell  = round.(vec(daily_grid_sell), digits=2)
    )

    out_path = joinpath(project_dir, "results", "battery_dispatch_solar_bud_uncertainty_7days_results.csv")
    CSV.write(out_path, df_results)
    println("\nSaved hourly decisions to: ", out_path)

    # ---------- Attach timestamps ----------
    df_solar = CSV.read(solar_path, DataFrame)
    fixed_strings = replace.(df_solar.datetime_utc, " " => "T")
    df_solar.datetime_utc = ZonedDateTime.(fixed_strings, dateformat"yyyy-mm-ddTHH:MM:SSzzzz")

    df_results[!, :datetime_utc] = df_solar.datetime_utc

    pacific = tz"America/Los_Angeles"
    df_results[!, :datetime_ca] = astimezone.(df_results.datetime_utc, pacific)

    df_results.datetime_utc_str = Dates.format.(df_results.datetime_utc, "yyyy-mm-dd HH:MM")
    df_results.datetime_ca_str  = Dates.format.(df_results.datetime_ca,  "yyyy-mm-dd HH:MM")

    select!(df_results,
        :datetime_utc_str => :datetime_utc,
        :datetime_ca_str  => :datetime_ca,
        :charge,
        :discharge,
        :grid_sell
    )

    out_path2 = joinpath(project_dir, "results", "battery_dispatch_solar_bud_uncertainty_with_timestamps.csv")
    CSV.write(out_path2, df_results)

    println("\nSaved WITH timestamps → ", out_path2)

end

main()


=== Solving day 1 (TRUE L1 Robust Solar) ===
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-20
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M2 Max
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 74 rows, 121 columns and 265 nonzeros
Model fingerprint: 0x67007739
Coefficient statistics:
  Matrix range     [2e-04, 1e+00]
  Objective range  [5e+00, 6e+01]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e-03, 5e+00]
Presolve removed 50 rows and 51 columns
Presolve time: 0.00s
Presolved: 24 rows, 70 columns, 93 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.4074547e+03   8.888884e+01   0.000000e+00      0s
       7    7.1886064e+02   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.00 seconds (0.00 work units)
Optimal objective  7.188606397e+02

User-callback calls 83, time in